# ClipRanker on Google Colab

Upload a video, and ClipRanker splits it into scenes, cuts each scene out as a clip, and ranks the clips by importance.

1. Choose **Runtime > Change runtime type > T4 GPU**.
2. Choose **Runtime > Run all**.
3. Setup takes about three minutes. Then the app opens in a new tab (cell 2). If the tab does not open, use cell 3 instead.

Source code and accuracy results: https://github.com/Kshitiz726/Semantic-Video-Synthesis-with-BERT

In [ ]:
# 1. Install (about 3 minutes)
!nvidia-smi -L || echo "No GPU found. It will still work, but slowly. Runtime > Change runtime type > T4 GPU"
!git clone -q https://github.com/Kshitiz726/Semantic-Video-Synthesis-with-BERT.git
%cd Semantic-Video-Synthesis-with-BERT
!pip install -q openai-whisper sentence-transformers ftfy librosa fastapi uvicorn python-multipart git+https://github.com/openai/CLIP.git
!python scripts/download_weights.py

In [ ]:
# 2. Start the ClipRanker app and open it
import subprocess, time, urllib.request
server = subprocess.Popen(["python", "-m", "app.server", "--port", "8000"],
                          stdout=open("server.log", "w"), stderr=subprocess.STDOUT)
print("Loading models...")
for _ in range(300):
    try:
        if b'"models_loaded":true' in urllib.request.urlopen("http://127.0.0.1:8000/api/health").read():
            break
    except Exception:
        pass
    time.sleep(2)
print("Ready.")
from google.colab import output
output.serve_kernel_port_as_window(8000, anchor_text="Open ClipRanker")

In [ ]:
# 3. Alternative without the app: upload a video here and get the ranked clips
import json
from google.colab import files
from IPython.display import Video, display

uploaded = files.upload()
name = next(iter(uploaded))
!python -m svs.pipeline "{name}" --out runs/colab

result = json.load(open("runs/colab/result.json"))
for s in sorted(result["scenes"], key=lambda s: s["rank"]):
    print(f"#{s['rank']:<3} scene {s['id']:<3} {s['start']:7.1f}s to {s['end']:7.1f}s  score {s['score']:.2f}")
best = min(result["scenes"], key=lambda s: s["rank"])
display(Video(f"runs/colab/clips/scene_{best['id']:03d}.mp4", embed=True, width=640))
print("All clips are in runs/colab/clips, the highlight reel is runs/colab/summary.mp4")